# Gradient boosting baseline on engineered features

Train a lightly tuned gradient boosting model on the engineered artifacts produced by notebook 08. This gives a clean single-model reference point before comparing stacking ensembles.

## 1. Notebook setup

### 1.1. Imports

In [8]:
import pickle
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from helper_functions.data_preprocessing import encode_label
from helper_functions.gradient_boosting_baseline import (
    add_candidate,
    build_model,
    build_relative_candidate,
    score_on_folds,
)

### 1.2. Run configuration

In [ ]:
# Engineered artifacts from notebook 08 and base hyperparameters from notebook 03.
ENGINEERED_CV_FOLDS     = '../data/tmp/08-engineered-cv-folds.pkl'
ENGINEERED_TRAIN_DATA   = '../data/tmp/08-engineered-train-data.csv'
ENGINEERED_TEST_DATA    = '../data/tmp/08-engineered-test-data.csv'
WINNING_HYPERPARAMETERS = '../data/results/03-winning-hyperparameters.pkl'

# Notebook outputs.
SEARCH_RESULTS_FILE     = '../data/results/09-gradient-boosting-search-results.pkl'
CROSS_VALIDATION_SCORES = '../data/results/09-gradient-boosting-scores.pkl'
FINAL_SUBMISSION_FILE   = '../data/submission.csv'

# Run controls.
RUN_PARAMETER_SEARCH = True
RUN_RERANK           = True
RUN_FULL_CV_ESTIMATE = True

# Fold controls. The engineered artifact contains 10 folds; None uses all.
TUNE_FOLD_LIMIT       = 3
TOP_K_FOR_FULL_RERANK = 10

# Optional sampling controls for faster CV estimates on large datasets.
USE_CV_SAMPLING           = True
CV_TRAIN_SAMPLE_FRAC      = 0.2
CV_VALIDATION_SAMPLE_FRAC = 0.2

# Stage-2 rerank controls (typically no sampling for a cleaner final ranking).
USE_SAMPLING_FOR_FULL_RERANK = False

# Wider random search around and beyond the previous optimum.
WIDE_SEARCH_RANDOM_CANDIDATES = 500
RANDOM_SEARCH_SEED = 315

# Logging controls for compact progress output.
STAGE1_PROGRESS_EVERY = 25

## 2. Load engineered artifacts

In [3]:
# Load pre-engineered CV folds for fold-safe model selection.
with open(ENGINEERED_CV_FOLDS, 'rb') as handle:
    engineered_folds = pickle.load(handle)

# Load the baseline winning HGB settings from notebook 03.
with open(WINNING_HYPERPARAMETERS, 'rb') as handle:
    winning_hgb_params = pickle.load(handle)

# Load full engineered train/test artifacts for final training and submission.
train_df = pd.read_csv(ENGINEERED_TRAIN_DATA)
test_df = pd.read_csv(ENGINEERED_TEST_DATA)

# Recreate label encoder for inverse transform on submission predictions.
raw_train = pd.read_csv(
    'https://media.githubusercontent.com/media/gperdrizet/fullstack-2605/'
    'refs/heads/main/data/student-health-risk-train.csv'
)
_, label_encoder = encode_label(raw_train['health_condition'])

print(f'Loaded {len(engineered_folds)} engineered folds')
print(f'Engineered train shape: {train_df.shape}')
print(f'Engineered test shape:  {test_df.shape}')
print(f'Winning HGB params from notebook 03: {winning_hgb_params}')

Loaded 10 engineered folds
Engineered train shape: (690088, 72)
Engineered test shape:  (295753, 72)
Winning HGB params from notebook 03: {'max_depth': 7, 'learning_rate': np.float64(0.05), 'max_iter': 200, 'max_features': np.float64(0.8)}


## 3. Define model and parameter search

### 3.1. Round 1 - wide sampled search

In [9]:
# Relative search specs around the notebook 03 winning point.
relative_candidate_specs = [
    {},
    {'learning_rate_mult': 0.75, 'max_iter_mult': 1.25, 'max_depth_delta': -1, 'max_features_mult': 0.8},
    {'learning_rate_mult': 1.00, 'max_iter_mult': 1.00, 'max_depth_delta': 0, 'max_features_mult': 1.0},
    {'learning_rate_mult': 1.00, 'max_iter_mult': 1.25, 'max_depth_delta': 0, 'max_features_mult': 1.0},
    {'learning_rate_mult': 1.25, 'max_iter_mult': 0.75, 'max_depth_delta': -1, 'max_features_mult': 0.8},
    {'learning_rate_mult': 0.75, 'max_iter_mult': 1.50, 'max_depth_delta': 1, 'max_features_mult': 0.8},
]

# Wider candidate pools to escape local neighborhoods when feature space changes.
wide_search_space = {
    'max_depth': [4, 5, 6, 7, 8, 9, 10],
    'learning_rate': [0.015, 0.025, 0.035, 0.05, 0.07, 0.1, 0.14],
    'max_iter': [120, 180, 240, 320, 420, 560],
    'max_features': [0.4, 0.55, 0.7, 0.85, 1.0],
    'l2_regularization': [0.0, 0.05, 0.1, 0.3, 0.8, 1.5],
    'min_samples_leaf': [10, 20, 40, 80, 120],
}

combinations = 1

for key, values in wide_search_space.items():
    combinations *= len(values)

print(f'Total possible combinations of wide search space: {combinations}')

parameter_candidates = []
seen_candidates = set()

# 1) Relative candidates around prior winning parameters.
for spec in relative_candidate_specs:
    parameter_candidates, seen_candidates = add_candidate(
        build_relative_candidate(winning_hgb_params, spec),
        parameter_candidates,
        seen_candidates,
    )

# 2) Randomly sampled broader candidates.
rng = np.random.default_rng(RANDOM_SEARCH_SEED)

for _ in range(WIDE_SEARCH_RANDOM_CANDIDATES):
    random_candidate = {
        'max_depth': int(rng.choice(wide_search_space['max_depth'])),
        'learning_rate': float(rng.choice(wide_search_space['learning_rate'])),
        'max_iter': int(rng.choice(wide_search_space['max_iter'])),
        'max_features': float(rng.choice(wide_search_space['max_features'])),
        'l2_regularization': float(rng.choice(wide_search_space['l2_regularization'])),
        'min_samples_leaf': int(rng.choice(wide_search_space['min_samples_leaf'])),
    }

    parameter_candidates, seen_candidates = add_candidate(
        random_candidate,
        parameter_candidates,
        seen_candidates
    )

# Ensure original winning parameters are present.
parameter_candidates, seen_candidates = add_candidate(
    dict(winning_hgb_params),
    parameter_candidates,
    seen_candidates
)

print(f'Generated {len(parameter_candidates)} unique parameter candidates for search.')

Total possible combinations of wide search space: 44100
Generated 41 unique parameter candidates for search.


In [ ]:
if RUN_PARAMETER_SEARCH:

    # Use all 10 folds by default. Set TUNE_FOLD_LIMIT to an int
    # for faster exploratory runs.
    if TUNE_FOLD_LIMIT is None or TUNE_FOLD_LIMIT <= 0:
        tune_folds = engineered_folds

    else:
        tune_folds = engineered_folds[:min(TUNE_FOLD_LIMIT, len(engineered_folds))]

    print(f'Using {len(tune_folds)} folds for parameter search.')

    search_rows = []

    total_candidates = len(parameter_candidates)
    print(f'Round 1: evaluating {total_candidates} candidates on {len(tune_folds)}/{len(engineered_folds)} folds')

    # Stage 1: sampled scoring on selected folds for broad candidate filtering.
    stage1_rows = []

    total_runtime = 0

    for candidate_index, params in enumerate(parameter_candidates, start=1):

        run_start = time.time()

        fold_scores = score_on_folds(
            tune_folds,
            params,
            use_sampling=USE_CV_SAMPLING,
            train_sample_fraction=CV_TRAIN_SAMPLE_FRAC,
            validation_sample_fraction=CV_VALIDATION_SAMPLE_FRAC,
        )

        row = {
            'params': params,
            'mean': float(np.mean(fold_scores)),
            'median': float(np.median(fold_scores)),
            'std': float(np.std(fold_scores)),
            'fold_scores': fold_scores,
        }

        stage1_rows.append(row)

        total_runtime += (time.time() - run_start) / 60

        candidate_rate = candidate_index / total_runtime

        # Compact progress updates instead of printing every candidate.
        if candidate_index % STAGE1_PROGRESS_EVERY == 0 or candidate_index == total_candidates:
            current_best = max(stage1_rows, key=lambda record: record['median'])
            print(
                f'{candidate_index:>3}/{total_candidates}: '
                f'best median BA: {current_best["median"]:.4f}, '
                f'total runtime: {total_runtime:.2f} min, '
                f'eval rate: {candidate_rate:.2f} candidates/min'
            )

    stage1_rows = sorted(stage1_rows, key=lambda row: row['median'], reverse=True)

    print(
        f'Round 1 complete: best median BA={stage1_rows[0]["median"]:.4f}.'
    )

    with open(SEARCH_RESULTS_FILE, 'wb') as handle:
        pickle.dump(stage1_rows, handle)

else:
    with open(SEARCH_RESULTS_FILE, 'rb') as handle:
        stage1_rows = pickle.load(handle)

best_params = stage1_rows[0]['params']
print(f'\nBest stage 1 tuned parameters: {best_params}')

Using 3 folds for parameter search.
Round 1: evaluating 41 candidates on 3/10 folds
  3/41: best median BA: 0.9497, total runtime: 1.63 min, eval rate: 1.85 candidates/min
  6/41: best median BA: 0.9497, total runtime: 3.20 min, eval rate: 1.87 candidates/min


### 3.2. Round 2 - full-fold rerank of top candidates

In [ ]:
# Stage 2: full-fold rerank for final selection.
if RUN_RERANK:

    top_stage1_rows = stage1_rows[:TOP_K_FOR_FULL_RERANK]

    print(
        f'Round 2: reranking top {len(top_stage1_rows)} candidates on '
        f'{len(engineered_folds)} full folds (sampling={USE_SAMPLING_FOR_FULL_RERANK}).'
    )

    rerank_rows = []

    for rerank_index, row in enumerate(top_stage1_rows, start=1):
        params = row['params']

        fold_scores = score_on_folds(
            engineered_folds,
            params,
            use_sampling=USE_SAMPLING_FOR_FULL_RERANK,
            train_sample_fraction=CV_TRAIN_SAMPLE_FRAC,
            validation_sample_fraction=CV_VALIDATION_SAMPLE_FRAC,
        )

        rerank_row = {
            'params': params,
            'mean': float(np.mean(fold_scores)),
            'median': float(np.median(fold_scores)),
            'std': float(np.std(fold_scores)),
            'fold_scores': fold_scores,
            'stage1_median': row['median'],
        }

        rerank_rows.append(rerank_row)

        print(
            f'  Round 2 candidate {rerank_index:>2}/{len(top_stage1_rows)}: '
            f'median BA={rerank_row["median"]:.4f} '
            f'(round-1 median={rerank_row["stage1_median"]:.4f})'
        )

    search_rows = sorted(rerank_rows, key=lambda row: row['median'], reverse=True)
    best_params = search_rows[0]['params']

    # Persist rerank results.
    search_payload = {
        'stage1_rows': stage1_rows,
        'rerank_rows': search_rows,
        'best_params': best_params,
        'config': {
            'tune_fold_limit': TUNE_FOLD_LIMIT,
            'top_k_for_full_rerank': TOP_K_FOR_FULL_RERANK,
            'use_cv_sampling': USE_CV_SAMPLING,
            'use_sampling_for_full_rerank': USE_SAMPLING_FOR_FULL_RERANK,
            'wide_search_random_candidates': WIDE_SEARCH_RANDOM_CANDIDATES,
            'stage1_progress_every': STAGE1_PROGRESS_EVERY,
        },
    }

    with open(SEARCH_RESULTS_FILE, 'wb') as handle:
        pickle.dump(search_payload, handle)

else:
    with open(SEARCH_RESULTS_FILE, 'rb') as handle:
        search_payload = pickle.load(handle)

In [ ]:
# Summarize each candidate by median score and fold-level uncertainty.
median_scores_df = pd.DataFrame({
    'median_score': [row['median'] for row in search_rows],
    '95% CI lower': [np.percentile(row['fold_scores'], 2.5) for row in search_rows],
    '95% CI upper': [np.percentile(row['fold_scores'], 97.5) for row in search_rows],
})

# Plot candidate rank (x-axis) versus median CV score with 95% fold interval.
plt.title('Hyperparameter combination scores')
plt.plot(list(range(len(median_scores_df))), median_scores_df['median_score'], color='black')
plt.fill_between(
    list(range(len(median_scores_df))),
    median_scores_df['95% CI lower'],
    median_scores_df['95% CI upper'],
    color='lightgray',
    alpha=0.5
)
plt.xlabel('Hyperparameter combination')
plt.ylabel('Median balanced accuracy')
plt.show()

## 4. Cross-validation performance estimate

In [ ]:
if RUN_FULL_CV_ESTIMATE:

    # Re-score the selected candidate on all engineered folds.
    full_fold_scores = score_on_folds(
        engineered_folds,
        best_params,
        use_sampling=USE_CV_SAMPLING,
        train_sample_fraction=CV_TRAIN_SAMPLE_FRAC,
        validation_sample_fraction=CV_VALIDATION_SAMPLE_FRAC,
    )

    # Store both metric summary and run settings for reproducibility.
    cv_results = {
        'params': best_params,
        'fold_scores': full_fold_scores,
        'mean': float(np.mean(full_fold_scores)),
        'median': float(np.median(full_fold_scores)),
        'std': float(np.std(full_fold_scores)),
        'used_sampling': USE_CV_SAMPLING,
        'train_sample_fraction': CV_TRAIN_SAMPLE_FRAC,
        'validation_sample_fraction': CV_VALIDATION_SAMPLE_FRAC,
    }

    with open(CROSS_VALIDATION_SCORES, 'wb') as handle:
        pickle.dump(cv_results, handle)

else:
    with open(CROSS_VALIDATION_SCORES, 'rb') as handle:
        cv_results = pickle.load(handle)

# Print a compact report for quick notebook comparisons.
print('Cross-validation summary')
print(f"Mean balanced accuracy:   {cv_results['mean']:.4f}")
print(f"Median balanced accuracy: {cv_results['median']:.4f}")
print(f"Std balanced accuracy:    {cv_results['std']:.4f}")
print(f"Used sampling:            {cv_results.get('used_sampling', False)}")

if cv_results.get('used_sampling', False):
    print(f"Train sample fraction:    {cv_results.get('train_sample_fraction', 1.0):.2f}")
    print(f"Validation sample fraction: {cv_results.get('validation_sample_fraction', 1.0):.2f}")

In [ ]:
plt.title('Cross-validation balanced accuracy distribution')
sns.boxplot(
    x=full_fold_scores,
    color='lightgray',
    boxprops={'facecolor': 'lightgray', 'edgecolor': 'black'},
    medianprops={'color': 'black', 'linewidth': 1.5},
    whiskerprops={'color': 'black'},
    capprops={'color': 'black'}
)
plt.xlabel('Balanced Accuracy')
plt.tight_layout()
plt.show()

## 5. Train final model and generate submission

In [ ]:
# Train one final model on the full engineered training artifact.
final_model = build_model(best_params, seed=315)
final_model.fit(train_df.drop('health_condition', axis=1), train_df['health_condition'])

# Predict on engineered test features (exclude id from model input).
test_features = test_df.drop('id', axis=1)
test_predictions = final_model.predict(test_features)

# Convert encoded predictions back to competition label strings.
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'health_condition': label_encoder.inverse_transform(test_predictions),
})

# Save submission in the standard project location.
submission_df.to_csv(FINAL_SUBMISSION_FILE, index=False)

print(f'Saved submission to {FINAL_SUBMISSION_FILE}')
print(submission_df['health_condition'].value_counts())

submission_df.head()